# Phase 4 scikit-learn Baseline

This notebook is a lightweight evaluation wrapper around the checked-in Phase 4 scripts. Replace `MANIFEST` with a private manifest before running. The checked-in example manifest is schema-only and will not train.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
MANIFEST = Path('path/to/private_manifest.json')
OUTPUT = ROOT / 'models' / 'baseline_sklearn'
if not MANIFEST.exists():
    raise FileNotFoundError('Set MANIFEST to a private Phase 4 dataset manifest.')
MANIFEST

## Session Holdout

Use this for same-glass supervised repeatability. This is not the Phase 3 local calibration workflow.

In [ ]:
import subprocess

subprocess.run([
    'python', 'scripts/train_baseline.py',
    '--manifest', str(MANIFEST),
    '--group-by', 'session_id',
    '--output-dir', str(OUTPUT),
], check=True)

## Generalization Checks

Run separate artifacts for glass, device, and browser holdouts. Reports must keep these metrics separate. The trainer evaluates repeated grouped holdouts inside each run.

In [ ]:
for group, name in [
    ('glass_id', 'baseline_sklearn_glass_holdout'),
    ('device_id', 'baseline_sklearn_device_holdout'),
    ('browser_id', 'baseline_sklearn_browser_holdout'),
]:
    subprocess.run([
        'python', 'scripts/train_baseline.py',
        '--manifest', str(MANIFEST),
        '--group-by', group,
        '--output-dir', str(ROOT / 'models' / name),
    ], check=True)

In [ ]:
import json

metrics_path = OUTPUT / 'metrics.json'
feature_schema_path = OUTPUT / 'feature_schema.json'
metrics = json.loads(metrics_path.read_text())
feature_schema = json.loads(feature_schema_path.read_text())
model_mae = metrics['regression']['mae_percent']
reference_mae = {
    name: value['mae_percent']
    for name, value in metrics['references'].items()
    if 'mae_percent' in value
}
deltas = {name: mae - model_mae for name, mae in reference_mae.items()}
(
    metrics['regression'],
    metrics['repeated_holdout']['regression'],
    deltas,
    feature_schema['feature_names'][:10],
)